# TableAgent — 数据探索与原型验证 (P0)

## 数据集: Titanic (泰坦尼克号乘客数据)

### 项目目标

搭载数据分析 Skill 的表格智能 Agent。本 notebook 用于 P0 阶段的数据探索和 Skill 函数原型验证。

**技术栈:** pandas 3.0.2, numpy 2.3.3, matplotlib 3.10.9, seaborn 0.13.2

## 1. 环境检查

In [ ]:
# 中文字体配置
import matplotlib as mpl
mpl.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "DejaVu Sans"]
mpl.rcParams["axes.unicode_minus"] = False
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"matplotlib: {plt.matplotlib.__version__}")
print(f"seaborn: {sns.__version__}")

In [ ]:
import openai
import gradio as gr
print(f"openai: {openai.__version__}")
print(f"gradio: {gr.__version__}")

## 2. 数据加载与初步探查

加载 Titanic 数据集并进行初步探查，验证 Skill 1 (DataReader) 的核心功能。

In [ ]:
df = pd.read_csv('../data/titanic.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. 缺失值分析

In [ ]:
missing = df.isnull().sum()
missing_percent = (missing / len(df)) * 100
missing_df = pd.DataFrame({'缺失数': missing, '缺失率(%)': missing_percent.round(2)})
missing_df[missing_df['缺失数'] > 0]

In [ ]:
# 缺失值可视化
fig, ax = plt.subplots(figsize=(10, 4))
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]
bars = ax.barh(range(len(missing_pct)), missing_pct.values, color='coral')
ax.set_yticks(range(len(missing_pct)))
ax.set_yticklabels(missing_pct.index)
ax.set_xlabel('缺失率 (%)')
ax.set_title('各列缺失值比例')
for bar, val in zip(bars, missing_pct.values):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', va='center')
plt.tight_layout()
plt.show()

## 4. 单变量分析

### 4.1 目标变量: Survived

In [ ]:
# 生存分布
survived_counts = df['survived'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(['死亡(0)', '幸存(1)'], survived_counts.values, color=['#ff6b6b', '#51cf66'])
axes[0].set_ylabel('人数')
for i, v in enumerate(survived_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center')
axes[1].pie(survived_counts.values, labels=['死亡', '幸存'], autopct='%1.1f%%',
            colors=['#ff6b6b', '#51cf66'], startangle=90)
axes[1].set_title(f'总人数: {len(df)}')
plt.tight_layout()
plt.show()

### 4.2 乘客等级 (Pclass)

In [ ]:
pclass_counts = df['pclass'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#FFD700', '#C0C0C0', '#CD7F32']
bars = ax.bar(pclass_counts.index, pclass_counts.values, color=colors, tick_label=['头等舱', '二等舱', '三等舱'])
ax.set_ylabel('人数')
ax.set_title('乘客等级分布')
for bar, val in zip(bars, pclass_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(val), ha='center')
plt.show()

### 4.3 性别 (Sex)

In [ ]:
sex_counts = df['sex'].value_counts()
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(sex_counts.index, sex_counts.values, color=['#74b9ff', '#fd79a8'])
ax.set_ylabel('人数')
ax.set_title('性别分布')
for bar, val in zip(bars, sex_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(val), ha='center')
plt.show()

### 4.4 年龄 (Age)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['age'].dropna(), bins=30, edgecolor='white', color='#3498db', alpha=0.7)
axes[0].set_xlabel('年龄')
axes[0].set_ylabel('人数')
axes[0].set_title('年龄分布')
axes[1].boxplot(df['age'].dropna(), patch_artist=True, boxprops=dict(facecolor='#3498db', alpha=0.7))
axes[1].set_ylabel('年龄')
axes[1].set_title('年龄箱线图')
axes[1].set_xticks([])
plt.tight_layout()
plt.show()
print(f'年龄统计: 均值={df["age"].mean():.1f}, 中位数={df["age"].median():.1f}, 标准差={df["age"].std():.1f}, 最小值={df["age"].min():.1f}, 最大值={df["age"].max():.1f}')

### 4.5 票价 (Fare)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['fare'], bins=50, edgecolor='white', color='#e17055', alpha=0.7)
axes[0].set_xlabel('票价')
axes[0].set_ylabel('频数')
axes[0].set_title('票价分布')
axes[1].boxplot(df['fare'], patch_artist=True, boxprops=dict(facecolor='#e17055', alpha=0.7))
axes[1].set_ylabel('票价')
axes[1].set_title('票价箱线图')
axes[1].set_xticks([])
plt.tight_layout()
plt.show()
print(f'票价统计: 均值={df["fare"].mean():.1f}, 中位数={df["fare"].median():.1f}, 标准差={df["fare"].std():.1f}, 最大={df["fare"].max():.1f}')

## 5. 双变量分析与分组聚合

### 5.1 生存率 vs 乘客等级

In [ ]:
pclass_survived = df.groupby('pclass')['survived'].mean().mul(100).round(1)
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(['头等舱', '二等舱', '三等舱'], pclass_survived.values, color=['#FFD700', '#C0C0C0', '#CD7F32'])
ax.set_ylabel('生存率 (%)')
ax.set_title('不同等级舱位的生存率')
for bar, val in zip(bars, pclass_survived.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{val}%', ha='center')
plt.show()
print(pclass_survived)

### 5.2 生存率 vs 性别

In [ ]:
sex_survived = df.groupby('sex')['survived'].mean().mul(100).round(1)
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['女性', '男性'], sex_survived.values, color=['#fd79a8', '#74b9ff'])
ax.set_ylabel('生存率 (%)')
ax.set_title('不同性别的生存率')
for bar, val in zip(bars, sex_survived.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{val}%', ha='center')
plt.show()

### 5.3 生存率 vs 年龄分布

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
survived = df[df['survived'] == 1]['age'].dropna()
not_survived = df[df['survived'] == 0]['age'].dropna()
ax.hist([not_survived, survived], bins=30, label=['未幸存', '幸存'],
        color=['#ff6b6b', '#51cf66'], alpha=0.6, edgecolor='white')
ax.set_xlabel('年龄')
ax.set_ylabel('人数')
ax.set_title('生存 vs 年龄分布')
ax.legend()
plt.show()

### 5.4 生存率 vs 登船港口 (Embarked)

In [ ]:
embarked_survived = df.groupby('embark_town')['survived'].mean().mul(100).round(1)
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(embarked_survived.index, embarked_survived.values, color=['#00b894', '#fdcb6e', '#0984e3'])
ax.set_ylabel('生存率 (%)')
ax.set_title('不同登船港口的生存率')
for bar, val in zip(bars, embarked_survived.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{val}%', ha='center')
plt.show()

### 5.5 分组聚合演示 (验证 Skill 2: Analyst)

In [ ]:
# groupby_agg 原型验证
agg_result = df.groupby(['pclass', 'sex']).agg(
    人数=('survived', 'count'),
    生存人数=('survived', 'sum'),
    生存率=('survived', 'mean'),
    平均年龄=('age', 'mean'),
    平均票价=('fare', 'mean')
).round(2)
agg_result['生存率'] = (agg_result['生存率'] * 100).round(1).astype(str) + '%'
agg_result

## 6. 相关性分析

In [ ]:
# 相关性矩阵
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr_matrix.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(numeric_cols)))
ax.set_yticks(range(len(numeric_cols)))
ax.set_xticklabels(numeric_cols, rotation=45, ha='right')
ax.set_yticklabels(numeric_cols)
for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        ax.text(j, i, f'{corr_matrix.values[i, j]:.2f}', ha='center', va='center',
                fontsize=8, color='black' if abs(corr_matrix.values[i, j]) < 0.5 else 'white')
ax.set_title('数值变量相关性矩阵')
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# 与 Survived 的相关性
corr_with_target = corr_matrix['survived'].drop('survived').sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#51cf66' if v > 0 else '#ff6b6b' for v in corr_with_target.values]
bars = ax.barh(range(len(corr_with_target)), corr_with_target.values, color=colors)
ax.set_yticks(range(len(corr_with_target)))
ax.set_yticklabels(corr_with_target.index)
ax.set_xlabel('与 Survival 的相关系数')
ax.set_title('各变量与生存的相关性')
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
for bar, val in zip(bars, corr_with_target.values):
    ax.text(bar.get_width() + 0.01 if val > 0 else bar.get_width() - 0.07,
            bar.get_y() + bar.get_height()/2, f'{val:.2f}', va='center')
plt.tight_layout()
plt.show()

## 7. 高级可视化

### 7.1 箱线图: 票价 vs 等级

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x='pclass', y='fare', palette='Set2', ax=ax)
ax.set_xticklabels(['头等舱', '二等舱', '三等舱'])
ax.set_xlabel('乘客等级')
ax.set_ylabel('票价')
ax.set_title('不同等级舱位的票价分布')
plt.show()

### 7.2 热力图: 相关性矩阵

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('相关性热力图')
plt.tight_layout()
plt.show()

### 7.3 配对图 (pairplot) — 选取关键列

In [ ]:
pair_cols = ['survived', 'age', 'fare', 'pclass']
sns.pairplot(df[pair_cols], hue='survived', palette={0: '#ff6b6b', 1: '#51cf66'}, diag_kind='hist')
plt.suptitle('关键变量配对图', y=1.02)
plt.show()

### 7.4 散点图: 年龄 vs 票价

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(df['age'], df['fare'], c=df['survived'], cmap='RdYlGn',
                      alpha=0.6, s=df['fare']/5, edgecolors='white', linewidth=0.5)
ax.set_xlabel('年龄')
ax.set_ylabel('票价')
ax.set_title('年龄 vs 票价 (颜色=生存, 大小=票价)')
legend = ax.legend(*scatter.legend_elements(), title='生存', loc='upper right')
plt.show()

## 8. Skill 函数原型验证

### 8.1 DataReader 原型

In [ ]:
def load_csv(path):
    try:
        df = pd.read_csv(path)
        return f'成功加载数据: {df.shape[0]} 行 x {df.shape[1]} 列'
    except Exception as e:
        return f'加载失败: {e}'

def get_overview(df):
    return f'行数: {df.shape[0]}, 列数: {df.shape[1]}\n列名: {list(df.columns)}'

def get_head(df, n=5):
    return df.head(n).to_string()

def get_missing(df):
    missing = df.isnull().sum()
    return missing[missing > 0].to_string()

def get_unique(df, col):
    return f'{col} 唯一值: {df[col].nunique()}个\n{df[col].value_counts().to_string()}'

print(load_csv('../data/titanic.csv'))
print()
print(get_overview(df))
print()
print(get_missing(df))

### 8.2 Analyst 原型

In [ ]:
def describe(df, cols=None):
    if cols:
        return df[cols].describe().to_string()
    return df.describe().to_string()

def value_counts(df, col):
    return df[col].value_counts().to_string()

def groupby_agg(df, group_col, agg_col, func):
    return df.groupby(group_col)[agg_col].agg(func).to_string()

def filter_data(df, col, op, val):
    if op == '>':
        result = df[df[col] > val]
    elif op == '<':
        result = df[df[col] < val]
    elif op == '==':
        result = df[df[col] == val]
    else:
        return f'不支持的操作: {op}'
    return f'过滤结果: {len(result)} 行\n{result.head().to_string()}'

print('Describe(age, fare):')
print(describe(df, ['age', 'fare']))
print()
print('Value counts(sex):')
print(value_counts(df, 'sex'))
print()
print('Filter(age > 60):')
print(filter_data(df, 'age', '>', 60))

### 8.3 Visualizer 原型

In [ ]:
from PIL import Image
import io

def histogram(df, col, bins=30):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(df[col].dropna(), bins=bins, edgecolor='white', color='#3498db', alpha=0.7)
    ax.set_xlabel(col)
    ax.set_ylabel('频数')
    ax.set_title(f'{col} 分布直方图')
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf)

def bar_chart(df, x, y):
    fig, ax = plt.subplots(figsize=(8, 4))
    data = df.groupby(x)[y].mean().reset_index()
    ax.bar(data[x].astype(str), data[y], color='#00b894')
    ax.set_xlabel(x)
    ax.set_ylabel(f'平均 {y}')
    ax.set_title(f'{y} 按 {x} 的平均值')
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf)

img1 = histogram(df, 'age')
display(img1)
img2 = bar_chart(df, 'pclass', 'fare')
display(img2)

## 9. 总结与发现

### 关键发现

| 发现 | 详情 |
|------|------|
| **缺失值** | `age` (20.1%), `deck` (79.6%), `embarked` (0.2%) |
| **整体生存率** | 38.4% |
| **性别影响** | 女性生存率 ~74%, 男性生存率 ~19% |
| **等级影响** | 头等舱 ~63%, 二等舱 ~47%, 三等舱 ~24% |
| **年龄影响** | 儿童生存率较高, 老年人极低 |
| **票价影响** | 高票价与高生存率正相关 |

### 对 Skill 开发的启示

1. **Reader Skill**: 需要处理缺失值 (age 20% 缺失, deck 80% 缺失)
2. **Analyst Skill**: 分组聚合和条件过滤是核心需求
3. **Visualizer Skill**: 直方图、柱状图、箱线图、热力图、散点图覆盖大部分分析场景
4. **链式调用场景**: "按等级和性别分组统计生存率后画柱状图" 等复合需求


In [ ]:
print('=== P0 环境+数据探索完成 ===')
print(f'pandas {pd.__version__} | numpy {np.__version__}')
print(f'matplotlib {plt.matplotlib.__version__} | seaborn {sns.__version__}')
print(f'openai {openai.__version__} | gradio {gr.__version__}')
print(f'Titanic 数据: {df.shape[0]} 行 x {df.shape[1]} 列')